In [ ]:
import os
os.environ['PYQTGRAPH_QT_LIB'] = 'PyQt5'
from fobench import Fiber
%gui qt
import matplotlib.pyplot as plt

# Signal and Wavefield Analysis
This notebooks delves a bit deeper into some of the Signal and Wavefield Analysis functionality that FoBench offers. For this tutorial we will mostly just generate plots of the results, however remember that you always have the option to return results by using `results=True` and suppress the plotting by setting `plot_mode=False`.
We will start by loading some data and apply basic pre-processing.



In [ ]:
das = Fiber("./example_data/aragon_h5/2024_07_11_05h15m16s_HDAS_2DRawData_Strain.h5", company="aragon")
das.differentiate()
das.filter(f_type="bandpass", freq=(1, 20))
das.trim(t0:=das.start_time+43, tf=t0+16)
das.plot()

## Signal-to-Noise Ratio and Peak-to-Peak Amplitude
First lets look at some simple features, and start with the Signal-to-Noise ratio and peak-to-peak amplitude. In FoBench the SNR is defined as the ratio between mean and standard deviation of a signal.

In [ ]:
das.SNR()

In [ ]:
das.p2p_amp()

## RMS Amplitude
Next up is the Root-Mean-Square (RMS) amplitude. It can either be computed on the full record or windowed and in both dimensions.

In [ ]:
das.rmsa() # dim="t", default, RMS-Distance Profile
das.rmsa(dim="d") # RMS-time series

In [ ]:
das.rmsa(window=0.5) # dim="t", default, window represents seconds
das.rmsa(window=5, dim="d") # window represents number of channels

## Frequency-Distance Profile
The fx-plot shows us the frequency content of the data either by a simple Fourier Amplitudes Spectrum or as Power Spectral Density.

In [ ]:
das.fx_plot() # mode="spectrum"
das.fx_plot(mode="psd")

## Autocorrelation Profile
Autocorrelations are able to provide a zero-offset reflectivity profile under the fibre-optic cable. For the basic case we only have to set the maximum lag or two-way-traveltime until which we want to compute the correlations. Lets go with 2 seconds.

In [ ]:
acf = das.acf_profile(max_lag=2, results=True)

We enhance the reflectivity features we can remove the average autocorrelation over the full cable by setting `deconvolve=True`. For longer cables it might be more appropriate to perform this in a sliding window by passing the `window_size` argument. In that case, for each channel the average autocorrelation of plus-minus`window_size//2` around the channel is substracted from the target channel autocorrelation function. 

In [ ]:
acf_deconv = das.acf_profile(max_lag=2, deconvolve=True, results=True)

Now lets visualise the difference between both results to see the influence the deconvolution had:

In [ ]:
diff = acf-acf_deconv
plt.imshow(diff, aspect="auto", vmin=-0.4, vmax=0.4, cmap="seismic")
plt.ylabel("Time Sample")
plt.xlabel("Distance Sample")

## Spatial Similarity
This method computes the pairwise maximum cross-correlation between channels. Similarly to te autocorrelation profile, we only have to pass a `maximum_lag` time in seconds:

In [ ]:
das.spatial_similarity(max_lag=5)

## Continuous Wavelet Transform
This function computes and plots the continuous wavelet transform of a 1D signal. It mainly wraps around the cwt function of PyWavelets so its best to look
directly look at the [documentation](https://pywavelets.readthedocs.io/en/latest/ref/cwt.html#pywt.cwt) for the explanations of the arguments. Here we will just take the data from a single channel and do a minimal example:

In [ ]:
from fobench.core.tools.signals import cwt
das.channel_plot(100)

In [ ]:
out = cwt(signal=das.get_data(100), 
          fs=das.sampling_rate,          
          times=das.times(time_type="unix"))